# x calibration walkthrough

This notebook walks through the x calibration of one synthetic resonator, step by step, using the same analysis functions that the pipeline's default steps call (`citkid/pipeline/default_steps.py`). The goal is to convert a complex S21 noise timestream into fractional frequency shift, $x = \delta f_r / f_r$.

| Step | Function | Pipeline step |
|---|---|---|
| Fit the readout gain | `xcal.gain.make_fr_spans`, `xcal.gain.fit_gain` | `make_fr_spans`, `fit_gain` |
| Remove the gain | `xcal.gain.remove_gain` | `rmv_gain_f`, `rmv_gain_t` |
| Fit the IQ circle | `xcal.circle.fit_iq_circle` | `fit_iq_circle` |
| Center and rotate | `xcal.xcal.nearest_index`, `xcal.circle.get_theta_phase_offset`, `xcal.circle.cent_rot_s21` | `get_idx_t`, `get_theta_phase_offset`, `center_f`, `center_t` |
| Convert to theta | `xcal.circle.convert_to_theta` | `get_thetaf`, `get_thetat` |
| Fit x(theta) | `xcal.xcal.calc_x`, `xcal.xcal.get_xcal_mask`, `xcal.xcal.fit_x_theta` | `get_xf`, `get_xcal_mask`, `fit_x_theta` |
| Convert the timestream to x | `np.polyval` | `get_xt` |

The data are synthetic: a gain sweep, a fine sweep, and a noise timestream with a known injected $x(t)$, so we can check the calibration at the end.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from citkid.res.generator import get_S21_from_xA
from citkid.xcal import gain, circle, xcal
from citkid.xcal.plot import plot_s21, plot_gain_fit, plot_circfit, plot_xcal
from citkid.signal.psd import get_psd, bin_psd

## 1. Synthetic data

**Resonator.** `get_S21_from_xA(f, p, x, A)` from `citkid.res.generator` gives the S21 of a (nonlinear) resonator with parameters `p = [fr, Qr, amp, phi, a]`, whose resonance frequency is shifted to `fr * (1 + x)` and whose depth is scaled by `1 + A`. It is deterministic: we supply $x$ and $A$ ourselves from a seeded `np.random.default_rng`. (The generator's own noise functions, `generate_noise` and `get_S21_vs_freq`, draw from numba's internal RNG, which `np.random.default_rng` does not seed, so we don't use them here.)

**Readout.** The resonator is multiplied by a readout gain: a quadratic in dB (amplifier slope and ripple) and a linear phase (constant phase plus cable delay $\tau$). This is the model that `fit_gain` fits. White IQ noise stands in for the amplifier noise.

In [ ]:
rng = np.random.default_rng(0)

### Resonator parameters
fr = 1e9                         # resonance frequency (Hz)
Qc, Qi = 30e3, 300e3
Qr = 1 / (1 / Qc + 1 / Qi)
amp = Qr / Qc
phi = 0.2                        # impedance-mismatch rotation
a = 0.3                          # nonlinearity parameter
p = np.array([fr, Qr, amp, phi, a])
linewidth = fr / Qr

### Readout gain and cable delay
gain_dB = (-12., 1.5, -0.8)      # dB, dB / MHz, dB / MHz^2 about fr
phase0 = 1.0                     # constant phase (rad)
tau = 50e-9                      # cable delay (s)

def readout_gain(f):
    u = (f - fr) / 1e6
    dB = gain_dB[0] + gain_dB[1] * u + gain_dB[2] * u ** 2
    return 10 ** (dB / 20) * np.exp(1j * (phase0 - 2 * np.pi * (f - fr) * tau))

def iq_noise(n, sigma):
    return sigma * (rng.normal(size = n) + 1j * rng.normal(size = n))

def sweep(f, sigma):
    zres = get_S21_from_xA(f, p, np.zeros(len(f)), np.zeros(len(f)))
    return (zres + iq_noise(len(f), sigma)) * readout_gain(f)

### Gain sweep (wide) and fine sweep (a few linewidths)
fg = np.linspace(fr - 2e6, fr + 2e6, 401)
zg = sweep(fg, 2e-3)
ff = np.linspace(fr - 3 * linewidth, fr + 3 * linewidth, 401)
zf = sweep(ff, 1e-3)

### Noise timestream at the tone frequency ft
# Put the tone at the minimum of |S21| of the noiseless sweep
zf_noiseless = get_S21_from_xA(ff, p, np.zeros(len(ff)), np.zeros(len(ff)))
ft = ff[np.argmin(np.abs(zf_noiseless))]
fs = 10e3                        # sample rate (Hz)
dt = 1 / fs
n = 100_000
t = np.arange(n) * dt
# Injected signal: 3 Hz sine wave plus white x noise
x_true = 5e-6 * np.sin(2 * np.pi * 3 * t) + rng.normal(0, 1e-6, n)
A_true = rng.normal(0, 1e-3, n)
zt = get_S21_from_xA(ft, p, x_true, A_true) + iq_noise(n, 1e-3)
zt = zt * readout_gain(ft)

print(f'linewidth = {linewidth / 1e3:.1f} kHz, ft - fr = {(ft - fr) / 1e3:.2f} kHz')
print(f'injected sine amplitude: x = 5e-6 = {5e-6 * Qr:.2f} / Qr')

The raw data: the gain sweep (wide), the fine sweep, and the noise timestream. In the IQ plane the cable delay makes the gain sweep trace an arc, and the fine sweep is a rotated, scaled circle.

In [ ]:
fig, axs = plot_s21(ff, zf, zt = zt, fg = fg, zg = zg, max_points = 5000)

## 2. Fit and remove the gain

`make_fr_spans(fres_all, qres_all)` builds one `(fr, fr / Qr)` span per resonance in the band. `fit_gain` cuts those spans (scaled by `span_mult`) out of the gain sweep, fits a 2nd-order polynomial to $|S_{21}|$ in dB and a 1st-order polynomial to the unwrapped phase, and returns the fit parameters plus the mask of points it used. The polynomials are in absolute frequency (Hz).

`remove_gain(f, z, p_amp, p_phase)` divides the gain out. It is applied to the fine sweep and to the timestream (at the tone frequency `ft`).

In [ ]:
fres_all = np.array([fr])        # in practice: every resonance in the band
qres_all = np.array([Qr])
fr_spans = gain.make_fr_spans(fres_all, qres_all)

span_mult = 10                   # the ts_analysis.yaml default is 1
p_amp, p_phase, gain_mask = gain.fit_gain(fg, zg, fr_spans,
                                          span_mult = span_mult)
fig, axs = plot_gain_fit(fg, zg, gain_mask, p_amp, p_phase)

zf_rmv = gain.remove_gain(ff, zf, p_amp, p_phase)
zt_rmv = gain.remove_gain(ft, zt, p_amp, p_phase)

`span_mult` sets how much of the resonance is cut. The template default, `span_mult = 1`, cuts only one linewidth ($f_r / Q_r$), so for this deep, high-Q resonator the wings bias the fit, most visibly the phase slope. Here we used 10, and the wings still tilt the phase fit slightly (the phase of each uncut segment is fit separately and the fits are averaged). The comparison below shows the error of the gain-removed fine sweep against the true resonator S21. The residual gain error is a small, nearly constant complex factor across the fine sweep, which the circle fit and rotation below absorb, so the x calibration is insensitive to it.

In [ ]:
for sm in [1, 3, 10]:
    pa, pp, gm = gain.fit_gain(fg, zg, fr_spans, span_mult = sm)
    err = gain.remove_gain(ff, zf, pa, pp) - zf_noiseless
    print(f'span_mult = {sm:2d}: {np.sum(~gm):3d} points cut, '
          f'max |error| on fine sweep = {np.max(np.abs(err)):.4f}')

In [ ]:
fig, axs = plot_s21(ff, zf_rmv, zt = zt_rmv, max_points = 5000)
axs[1].plot(1, 0, 'xk', label = 'off resonance (1, 0)')
axs[1].legend(framealpha = 1, loc = [1.01, 0.])

## 3. Fit the IQ circle

`fit_iq_circle(z, mask)` fits a circle to the gain-removed fine sweep and returns its center (`origin`, complex) and `radius`. `mask = None` (the template default, `circ_mask: null`) uses every point. Pass a boolean mask to fit only part of the loop.

In [ ]:
circ_mask = None
origin, radius = circle.fit_iq_circle(zf_rmv, mask = circ_mask)
print(f'origin = {origin:.4f}, radius = {radius:.4f}')
fig, ax = plot_circfit(zf_rmv, origin, radius, zt = zt_rmv, mask = circ_mask,
                       max_points = 5000)

## 4. Center, rotate, and convert to theta

- `nearest_index(ff, ft)` is the fine-sweep index closest to the tone, `idx_t`.
- `get_theta_phase_offset(zt_rmv, origin)` is the angle of the median of the centered timestream. Rotating by it puts the timestream near theta = 0.
- `cent_rot_s21(z, origin, theta_phase_offset)` centers and rotates both the fine sweep and the timestream.
- `convert_to_theta` takes the angle. For the fine sweep it unwraps, and `idx_t` removes any $2\pi$ offset at the tone so the sweep and timestream thetas agree. The timestream is not unwrapped (it stays near 0).

In [ ]:
idx_t = xcal.nearest_index(ff, ft)
theta_phase_offset = circle.get_theta_phase_offset(zt_rmv, origin)

zf_cent = circle.cent_rot_s21(zf_rmv, origin, theta_phase_offset)
zt_cent = circle.cent_rot_s21(zt_rmv, origin, theta_phase_offset)

thetaf = circle.convert_to_theta(zf_cent, unwrap = True, idx_t = idx_t)
thetat = circle.convert_to_theta(zt_cent, unwrap = False)
print(f'theta_phase_offset = {theta_phase_offset:.4f} rad, '
      f'median(thetat) = {np.median(thetat):.1e} rad')

fig, axs = plt.subplots(1, 2, figsize = [9, 3.5], layout = 'tight')
axs[0].plot((ff - ft) / 1e3, thetaf, '.', label = 'fine sweep')
axs[0].axhspan(thetat.min(), thetat.max(), color = 'C1', alpha = 0.3,
               label = 'timestream theta range')
axs[0].axvline(0, color = 'k', ls = ':', label = 'ft')
axs[0].set(xlabel = '(f - ft) (kHz)', ylabel = 'theta (rad)')
axs[0].legend(loc = 'lower left')
axs[1].plot(zf_cent.real, zf_cent.imag, '.', label = 'fine sweep')
show = np.arange(0, n, 20)
axs[1].plot(zt_cent[show].real, zt_cent[show].imag, '.', ms = 2,
            label = 'timestream')
axs[1].plot(0, 0, '+k')
axs[1].set(xlabel = 'I', ylabel = 'Q', aspect = 'equal',
           adjustable = 'datalim')
axs[1].legend()

## 5. Fit x(theta)

Each fine-sweep point has $x_f = 1 - f / f_t$ (`calc_x(ff, ft)`): the fractional resonance shift at which the tone would see the S21 that the sweep measured at $f$. Fitting $x_f$ against $\theta_f$ gives the calibration $x(\theta)$.

The fit only needs to be good over the range the timestream explores. `get_xcal_mask` keeps the fine-sweep points whose theta lies within the timestream's theta range, after cutting timestream points more than `std_cutoff` standard deviations from the mean (glitches). It then extends the range by `idx0_offset` / `idx1_offset` points on the low- / high-frequency side. `fit_x_theta` fits a polynomial; the pipeline uses degree 3. The values below are the `ts_analysis.yaml` template defaults.

In [ ]:
xcal_idx0_offset = 3
xcal_idx1_offset = 9
xcal_std_cutoff = 16.0
poly_x_deg = 3

xf = xcal.calc_x(ff, ft)
xcal_mask = xcal.get_xcal_mask(ff, thetaf, thetat, xcal_idx0_offset,
                               xcal_idx1_offset, xcal_std_cutoff)
poly_x = xcal.fit_x_theta(thetaf, xf, xcal_mask, poly_x_deg)

resid = xf[xcal_mask] - np.polyval(poly_x, thetaf[xcal_mask])
print(f'{xcal_mask.sum()} fine-sweep points in the fit, theta from '
      f'{thetaf[xcal_mask].min():.2f} to {thetaf[xcal_mask].max():.2f} rad')
print(f'rms fit residual = {np.std(resid):.1e} '
      f'(x range of the fit = {np.ptp(xf[xcal_mask]):.1e})')
fig, axs = plot_xcal(thetaf, xf, zf_cent, xcal_mask, poly_x, thetat = thetat,
                     zt_cent = zt_cent, std_cutoff = xcal_std_cutoff,
                     max_points = 5000)

## 6. Convert the timestream to x

The calibrated timestream is $x_t = $ `np.polyval(poly_x, thetat)` (the `get_xt` calibration step). Because the data are synthetic we can compare it with the injected $x(t)$. The residual is the IQ (amplifier) noise projected onto x.

In [ ]:
xt = np.polyval(poly_x, thetat)

slope, offset = np.polyfit(x_true, xt, 1)
print(f'xt vs x_true: slope = {slope:.4f}, offset = {offset:.1e}')
print(f'rms(xt - x_true) = {np.std(xt - x_true):.1e}, '
      f'rms(x_true) = {np.std(x_true):.1e}')

fig, axs = plt.subplots(1, 2, figsize = [10, 3.5], layout = 'tight')
sl = slice(0, int(1 / dt))       # first second
axs[0].plot(t[sl], x_true[sl] * 1e6, label = 'injected x')
axs[0].plot(t[sl], xt[sl] * 1e6, alpha = 0.6, label = 'calibrated xt')
axs[0].set(xlabel = 'Time (s)', ylabel = r'x ($10^{-6}$)')
axs[0].legend(loc = 'upper right')

f_psd, sxx = get_psd(xt, dt, get_frequencies = True)
sxx_true = get_psd(x_true, dt)
sxx_resid = get_psd(xt - x_true, dt)
f_b, sxx_b, sxx_true_b, sxx_resid_b = bin_psd(
    f_psd, [f_psd, sxx, sxx_true, sxx_resid], nbins = 200, fmin = 1)
axs[1].loglog(f_b[1:], sxx_true_b[1:], label = 'injected x')
axs[1].loglog(f_b[1:], sxx_b[1:], '--', label = 'calibrated xt')
axs[1].loglog(f_b[1:], sxx_resid_b[1:], label = 'xt - injected x')
axs[1].set(xlabel = 'Frequency (Hz)', ylabel = r'$S_{xx}$ (1/Hz)')
axs[1].legend()

The calibrated timestream matches the injected one (slope close to 1), and its white level matches the injected $S_{xx} = 2\sigma_x^2 / f_s = 2 \times 10^{-16}$ / Hz.

## Next: many rows with the pipeline

In practice these steps run for every tone of a measurement through `citkid.pipeline`. The analysis steps (`make_fr_spans`, `fit_gain`, `fit_iq_circle`, `get_idx_t`, `get_theta_phase_offset`, `get_xcal_mask`, `fit_x_theta`) come from the `ts_analysis.yaml` template, and the calibration steps (`rmv_gain_*`, `center_*`, `get_theta*`, `get_xf`, `get_xt`, PSDs) from the `cal-ts.yaml` template. The `plot_*` functions used above are the pipeline's default review plots. See `notebooks/pipeline.ipynb`.